In [1]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, rankdata


SCHEME_DISPLAY_NAMES = {
    "rand": "Rand",
    "uniform": "Unifrnd (0,1)",
    "beta_3_2": "Betarnd (3,2)",
    "beta_2_5_2_5": "Betarnd (2.5, 2.5)",
    "lognorm_0_0_5": "Lognrnd (0, 0.5)",
    "lognorm_0_0_67": "Lognrnd (0, 0.67)",
    "exprnd_0_5": "Exprnd (0.5)",
    "rayleigh_0_4": "Raylrnd (0.4)",
    "weibull_1_1": "Wblrnd (1,1)",
    "lhsdesign": "lhsdesign",
}

ALGORITHM_DISPLAY_NAMES = {
    "SGuA2": "SGuA2",
    "CMAES": "CMA-ES",
    "LSHADE": "L-SHADE",
}


# =========================================================
# ORC PROBLEM
# =========================================================

class ORCProblem:
    def __init__(self, penalty_weight=1e6):
        self.n_relays = 6
        self.penalty_weight = penalty_weight
        self.tol = 1e-12

        self.Ikd = np.array([1263.0, 5639.0, 5639.0, 5639.0, 5639.0, 5639.0], dtype=float)

        self.td_lb = np.array([0.2, 0.2, 0.2, 0.2, 0.2, 0.2], dtype=float)
        self.td_ub = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)

        self.ip_lb = np.array([117.152, 523.0, 400.0, 500.0, 600.0, 400.0], dtype=float)
        self.ip_ub = np.array([128.650, 575.0, 420.0, 510.0, 610.0, 420.0], dtype=float)

        self.ti_min = 1.0
        self.ti_max = 2.2

        self.lb = np.concatenate([self.td_lb, self.ip_lb])
        self.ub = np.concatenate([self.td_ub, self.ip_ub])

    def split_variables(self, X):
        X = np.atleast_2d(X)
        td = X[:, :6]
        Ip = X[:, 6:]
        return td, Ip

    def compute_ti(self, X):
        X = np.atleast_2d(X)
        td, Ip = self.split_variables(X)
        ratio = self.Ikd / Ip
        denom = np.power(ratio, 0.02) - 1.0
        denom = np.where(np.abs(denom) < 1e-12, 1e-12, denom)
        ti = 0.14 * td / denom
        return ti

    def evaluate(self, X):
        X = np.atleast_2d(X)
        td, Ip = self.split_variables(X)
        ti = self.compute_ti(X)
        objective = np.sum(ti, axis=1)

        td_low_v = np.maximum(0.0, self.td_lb - td)
        td_high_v = np.maximum(0.0, td - self.td_ub)
        ip_low_v = np.maximum(0.0, self.ip_lb - Ip)
        ip_high_v = np.maximum(0.0, Ip - self.ip_ub)
        ti_low_v = np.maximum(0.0, self.ti_min - ti)
        ti_high_v = np.maximum(0.0, ti - self.ti_max)

        c12 = np.maximum(0.0, 0.3 - (ti[:, 0] - ti[:, 1]))
        c23 = np.maximum(0.0, 0.3 - (ti[:, 1] - ti[:, 2]))
        c24 = np.maximum(0.0, 0.3 - (ti[:, 1] - ti[:, 3]))
        c25 = np.maximum(0.0, 0.3 - (ti[:, 1] - ti[:, 4]))
        c26 = np.maximum(0.0, 0.3 - (ti[:, 1] - ti[:, 5]))

        penalty = (
            np.sum(td_low_v ** 2, axis=1)
            + np.sum(td_high_v ** 2, axis=1)
            + np.sum(ip_low_v ** 2, axis=1)
            + np.sum(ip_high_v ** 2, axis=1)
            + np.sum(ti_low_v ** 2, axis=1)
            + np.sum(ti_high_v ** 2, axis=1)
            + c12 ** 2 + c23 ** 2 + c24 ** 2 + c25 ** 2 + c26 ** 2
        )

        penalized_fitness = objective + self.penalty_weight * penalty

        feasible = (
            (np.sum(td_low_v, axis=1) <= self.tol)
            & (np.sum(td_high_v, axis=1) <= self.tol)
            & (np.sum(ip_low_v, axis=1) <= self.tol)
            & (np.sum(ip_high_v, axis=1) <= self.tol)
            & (np.sum(ti_low_v, axis=1) <= self.tol)
            & (np.sum(ti_high_v, axis=1) <= self.tol)
            & (c12 <= self.tol)
            & (c23 <= self.tol)
            & (c24 <= self.tol)
            & (c25 <= self.tol)
            & (c26 <= self.tol)
        )
        return penalized_fitness, objective, feasible, ti

    def check_constraints_from_ti(self, ti):
        ti = np.asarray(ti, dtype=float)
        return {
            "t1_range": self.ti_min <= ti[0] <= self.ti_max,
            "t2_range": self.ti_min <= ti[1] <= self.ti_max,
            "t3_range": self.ti_min <= ti[2] <= self.ti_max,
            "t4_range": self.ti_min <= ti[3] <= self.ti_max,
            "t5_range": self.ti_min <= ti[4] <= self.ti_max,
            "t6_range": self.ti_min <= ti[5] <= self.ti_max,
            "t1_minus_t2": ti[0] - ti[1],
            "t2_minus_t3": ti[1] - ti[2],
            "t2_minus_t4": ti[1] - ti[3],
            "t2_minus_t5": ti[1] - ti[4],
            "t2_minus_t6": ti[1] - ti[5],
            "c12_ok": (ti[0] - ti[1]) >= 0.3,
            "c23_ok": (ti[1] - ti[2]) >= 0.3,
            "c24_ok": (ti[1] - ti[3]) >= 0.3,
            "c25_ok": (ti[1] - ti[4]) >= 0.3,
            "c26_ok": (ti[1] - ti[5]) >= 0.3,
        }


# =========================================================
# SGuA
# =========================================================

class SGuA:
    def __init__(
        self,
        problem,
        variant="SGuA1",
        P=40,
        G=500,
        branch_mode="linear",
        similarity_tol=1e-3,
        use_orc_guide_seed=False,
        init_scheme="rand",
        r_mode="vector",
        seed=123,
    ):
        self.problem = problem
        self.variant = variant
        self.P = P
        self.G = G
        self.D = len(problem.lb)
        self.branch_mode = branch_mode
        self.similarity_tol = similarity_tol
        self.use_orc_guide_seed = use_orc_guide_seed
        self.init_scheme = init_scheme
        self.r_mode = r_mode

        self.lb = problem.lb.copy()
        self.ub = problem.ub.copy()
        self.span = self.ub - self.lb
        self.max_distance = np.linalg.norm(self.span)
        self.rng = np.random.default_rng(seed)

        self.fe_count = 0
        self.gbest_X = np.zeros(self.D, dtype=float)
        self.gbest_penalized = np.inf
        self.gbest_objective = np.inf
        self.gbest_ti = None
        self.best_feasible_X = None
        self.best_feasible_obj = np.inf
        self.best_feasible_ti = None
        self.best_feasible_curve = np.full(G, np.nan, dtype=float)

    def evaluate_population(self, X):
        penalized, objective, feasible, ti = self.problem.evaluate(X)
        self.fe_count += X.shape[0]
        return penalized, objective, feasible, ti

    def update_bests(self, X, penalized, objective, feasible, ti):
        idx_pen = np.argmin(penalized)
        if penalized[idx_pen] < self.gbest_penalized:
            self.gbest_penalized = penalized[idx_pen]
            self.gbest_objective = objective[idx_pen]
            self.gbest_X = X[idx_pen].copy()
            self.gbest_ti = ti[idx_pen].copy()

        feasible_indices = np.where(feasible)[0]
        if feasible_indices.size > 0:
            local_feasible_obj = objective[feasible_indices]
            best_local_idx = feasible_indices[np.argmin(local_feasible_obj)]
            if objective[best_local_idx] < self.best_feasible_obj:
                self.best_feasible_obj = objective[best_local_idx]
                self.best_feasible_X = X[best_local_idx].copy()
                self.best_feasible_ti = ti[best_local_idx].copy()

    def select_best(self, X):
        penalized, objective, feasible, ti = self.evaluate_population(X)
        self.update_bests(X, penalized, objective, feasible, ti)
        idx = np.argsort(penalized)[: self.P]
        return X[idx].copy()

    def _positive_to_unit(self, x):
        return x / (1.0 + x)

    def _lhs_1d(self, n):
        perm = self.rng.permutation(n)
        return (perm + self.rng.random(n)) / n

    def _draw_r_batch(self, n):
        scheme = self.init_scheme.lower()
        size = n if self.r_mode == "scalar" else (n, self.D)

        if scheme == "rand":
            r = self.rng.random(size)
        elif scheme == "uniform":
            r = self.rng.uniform(0.0, 1.0, size=size)
        elif scheme == "beta_3_2":
            r = self.rng.beta(3.0, 2.0, size=size)
        elif scheme == "beta_2_5_2_5":
            r = self.rng.beta(2.5, 2.5, size=size)
        elif scheme == "lognorm_0_0_5":
            r = self._positive_to_unit(self.rng.lognormal(mean=0.0, sigma=0.5, size=size))
        elif scheme == "lognorm_0_0_67":
            r = self._positive_to_unit(self.rng.lognormal(mean=0.0, sigma=0.67, size=size))
        elif scheme == "exprnd_0_5":
            r = self._positive_to_unit(self.rng.exponential(scale=0.5, size=size))
        elif scheme == "rayleigh_0_4":
            r = self._positive_to_unit(self.rng.rayleigh(scale=0.4, size=size))
        elif scheme == "weibull_1_1":
            r = self._positive_to_unit(self.rng.weibull(a=1.0, size=size))
        elif scheme == "lhsdesign":
            if self.r_mode == "scalar":
                r = self._lhs_1d(n)
            else:
                r = np.column_stack([self._lhs_1d(n) for _ in range(self.D)])
        else:
            raise ValueError(f"Unknown init_scheme: {self.init_scheme}")

        eps = 1e-12
        return np.clip(r, eps, 1.0 - eps)

    def sowing_sapling(self):
        garden = [self.ub.copy(), self.lb.copy()]
        if self.use_orc_guide_seed and len(garden) < self.P:
            target_t = np.array([1.6, 1.3, 1.0, 1.0, 1.0, 1.0], dtype=float)
            ip_seed = self.problem.ip_lb.copy()
            denom = np.power(self.problem.Ikd / ip_seed, 0.02) - 1.0
            td_seed = target_t * denom / 0.14
            garden.append(np.clip(np.concatenate([td_seed, ip_seed]), self.lb, self.ub))

        k = 2
        while len(garden) < self.P:
            seg_ids = np.minimum((np.arange(self.D) * k) // self.D, k - 1)
            masks = list(range(1, (2 ** k) - 1))
            masks = masks[: self.P - len(garden)]
            if len(masks) == 0:
                break
            r_batch = self._draw_r_batch(len(masks))
            for idx_mask, mask in enumerate(masks):
                bits = np.array([(mask >> s) & 1 for s in range(k)], dtype=int)
                if self.r_mode == "scalar":
                    r = float(r_batch[idx_mask])
                    coeff = np.where(bits[seg_ids] == 1, r, 1.0 - r)
                else:
                    r = r_batch[idx_mask]
                    coeff = np.where(bits[seg_ids] == 1, r, 1.0 - r)
                x = self.lb + self.span * coeff
                garden.append(np.clip(x, self.lb, self.ub))
            k += 1
        return np.array(garden[: self.P], dtype=float)

    def mating(self, G):
        perm = self.rng.permutation(len(G))
        offspring = []
        for p in range(0, len(G) - 1, 2):
            s1 = G[perm[p]].copy()
            s2 = G[perm[p + 1]].copy()
            distance = np.linalg.norm(s1 - s2)
            pm = max(0.0, 1.0 - distance / (self.max_distance + 1e-12))
            child1 = s1.copy()
            child2 = s2.copy()
            for j in range(self.D):
                if self.rng.random() < pm:
                    child1[j], child2[j] = child2[j], child1[j]
            offspring.append(np.clip(child1, self.lb, self.ub))
            offspring.append(np.clip(child2, self.lb, self.ub))
        if len(G) % 2 == 1:
            offspring.append(G[perm[-1]].copy())
        return np.array(offspring[: self.P], dtype=float)

    def branching_probability(self, distance_index):
        if self.branch_mode == "linear":
            return 1.0 - 1.0 / (distance_index + 1.0)
        return 1.0 - np.exp(-distance_index)

    def branching(self, G):
        offspring = []
        for s in G:
            child = s.copy()
            i = self.rng.integers(0, self.D)
            for j in range(self.D):
                if j == i:
                    continue
                dist_idx = abs(j - i)
                if self.rng.random() < self.branching_probability(dist_idx):
                    child[j] = child[j] + self.rng.random() * (child[i] - child[j])
            offspring.append(np.clip(child, self.lb, self.ub))
        return np.array(offspring[: self.P], dtype=float)

    def vaccinating(self, G):
        perm = self.rng.permutation(len(G))
        offspring = []
        for p in range(0, len(G) - 1, 2):
            s1 = G[perm[p]].copy()
            s2 = G[perm[p + 1]].copy()
            norm_diff = np.abs(s1 - s2) / (self.span + 1e-12)
            dis = np.mean(norm_diff)
            child1 = s1.copy()
            child2 = s2.copy()
            if dis > self.rng.random():
                eq_mask = norm_diff <= self.similarity_tol
                rand1 = self.rng.uniform(self.lb, self.ub)
                rand2 = self.rng.uniform(self.lb, self.ub)
                child1 = np.where(eq_mask, s1, rand1)
                child2 = np.where(eq_mask, s2, rand2)
            offspring.append(np.clip(child1, self.lb, self.ub))
            offspring.append(np.clip(child2, self.lb, self.ub))
        if len(G) % 2 == 1:
            offspring.append(G[perm[-1]].copy())
        return np.array(offspring[: self.P], dtype=float)

    def opt_sgua1(self):
        G = self.sowing_sapling()
        pen, obj, fea, ti = self.evaluate_population(G)
        self.update_bests(G, pen, obj, fea, ti)
        if self.best_feasible_X is not None:
            self.best_feasible_curve[0] = self.best_feasible_obj
        for g in range(1, self.G):
            G1 = self.mating(G)
            G2 = self.branching(G)
            G3 = self.vaccinating(G)
            G = self.select_best(np.vstack([G1, G2, G3, G]))
            if self.best_feasible_X is not None:
                self.best_feasible_curve[g] = self.best_feasible_obj
        return self.finalize_result()

    def opt_sgua2(self):
        G = self.sowing_sapling()
        pen, obj, fea, ti = self.evaluate_population(G)
        self.update_bests(G, pen, obj, fea, ti)
        if self.best_feasible_X is not None:
            self.best_feasible_curve[0] = self.best_feasible_obj
        for g in range(1, self.G):
            G1 = self.mating(G)
            pen1, obj1, fea1, ti1 = self.evaluate_population(G1)
            self.update_bests(G1, pen1, obj1, fea1, ti1)
            Gm = self.select_best(np.vstack([G1, G]))
            G2 = self.branching(Gm)
            pen2, obj2, fea2, ti2 = self.evaluate_population(G2)
            self.update_bests(G2, pen2, obj2, fea2, ti2)
            Gb = self.select_best(np.vstack([G2, Gm]))
            G3 = self.vaccinating(Gb)
            pen3, obj3, fea3, ti3 = self.evaluate_population(G3)
            self.update_bests(G3, pen3, obj3, fea3, ti3)
            G = self.select_best(np.vstack([G3, Gb]))
            if self.best_feasible_X is not None:
                self.best_feasible_curve[g] = self.best_feasible_obj
        return self.finalize_result()

    def finalize_result(self):
        if self.best_feasible_X is not None:
            final_X = self.best_feasible_X.copy()
            final_ti = self.best_feasible_ti.copy()
            final_obj = self.best_feasible_obj
            feasible_flag = True
        else:
            final_X = self.gbest_X.copy()
            final_ti = self.gbest_ti.copy() if self.gbest_ti is not None else np.full(6, np.nan)
            final_obj = self.gbest_objective
            feasible_flag = False
        return {
            "x": final_X,
            "td": final_X[:6],
            "Ip": final_X[6:],
            "Ti": final_ti,
            "best_fitness": final_obj,
            "feasible": feasible_flag,
            "function_evaluations": self.fe_count,
            "feasible_curve": self.best_feasible_curve.copy(),
            "penalized_fitness": self.gbest_penalized,
            "init_scheme": self.init_scheme,
            "algorithm": self.variant,
            "method_key": f"{self.variant}-{self.init_scheme}",
        }

    def opt(self):
        if self.variant == "SGuA1":
            return self.opt_sgua1()
        if self.variant == "SGuA2":
            return self.opt_sgua2()
        raise ValueError("variant must be either 'SGuA1' or 'SGuA2'")


# =========================================================
# COMMON BASE FOR CMA-ES / L-SHADE
# =========================================================

class OptimizerBase:
    def __init__(self, problem, algorithm_name, P=100, G=1000, seed=123):
        self.problem = problem
        self.algorithm_name = algorithm_name
        self.P = P
        self.G = G
        self.D = len(problem.lb)
        self.lb = problem.lb.copy()
        self.ub = problem.ub.copy()
        self.span = self.ub - self.lb
        self.rng = np.random.default_rng(seed)

        self.fe_count = 0
        self.gbest_X = np.zeros(self.D, dtype=float)
        self.gbest_penalized = np.inf
        self.gbest_objective = np.inf
        self.gbest_ti = None
        self.best_feasible_X = None
        self.best_feasible_obj = np.inf
        self.best_feasible_ti = None
        self.best_feasible_curve = np.full(G, np.nan, dtype=float)

    def evaluate_population(self, X):
        penalized, objective, feasible, ti = self.problem.evaluate(X)
        self.fe_count += X.shape[0]
        return penalized, objective, feasible, ti

    def update_bests(self, X, penalized, objective, feasible, ti):
        idx_pen = np.argmin(penalized)
        if penalized[idx_pen] < self.gbest_penalized:
            self.gbest_penalized = penalized[idx_pen]
            self.gbest_objective = objective[idx_pen]
            self.gbest_X = X[idx_pen].copy()
            self.gbest_ti = ti[idx_pen].copy()
        feasible_idx = np.where(feasible)[0]
        if feasible_idx.size > 0:
            best_idx = feasible_idx[np.argmin(objective[feasible_idx])]
            if objective[best_idx] < self.best_feasible_obj:
                self.best_feasible_obj = objective[best_idx]
                self.best_feasible_X = X[best_idx].copy()
                self.best_feasible_ti = ti[best_idx].copy()

    def clip(self, X):
        return np.clip(X, self.lb, self.ub)

    def maybe_store_curve(self, g):
        if self.best_feasible_X is not None:
            self.best_feasible_curve[g] = self.best_feasible_obj

    def finalize_result(self):
        if self.best_feasible_X is not None:
            final_X = self.best_feasible_X.copy()
            final_ti = self.best_feasible_ti.copy()
            final_obj = self.best_feasible_obj
            feasible_flag = True
        else:
            final_X = self.gbest_X.copy()
            final_ti = self.gbest_ti.copy() if self.gbest_ti is not None else np.full(6, np.nan)
            final_obj = self.gbest_objective
            feasible_flag = False
        return {
            "x": final_X,
            "td": final_X[:6],
            "Ip": final_X[6:],
            "Ti": final_ti,
            "best_fitness": final_obj,
            "feasible": feasible_flag,
            "function_evaluations": self.fe_count,
            "feasible_curve": self.best_feasible_curve.copy(),
            "penalized_fitness": self.gbest_penalized,
            "algorithm": self.algorithm_name,
            "method_key": self.algorithm_name,
        }


# =========================================================
# CMA-ES (self-contained bounded adaptation)
# =========================================================

class CMAES(OptimizerBase):
    def __init__(self, problem, P=100, G=1000, seed=123, sigma0=0.30):
        super().__init__(problem, algorithm_name="CMAES", P=P, G=G, seed=seed)
        self.sigma0 = sigma0

    def opt(self):
        n = self.D
        lam = self.P
        mu = lam // 2
        weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
        weights = weights / np.sum(weights)
        mueff = (np.sum(weights) ** 2) / np.sum(weights ** 2)

        cc = (4 + mueff / n) / (n + 4 + 2 * mueff / n)
        cs = (mueff + 2) / (n + mueff + 5)
        c1 = 2 / ((n + 1.3) ** 2 + mueff)
        cmu = min(1 - c1, 2 * (mueff - 2 + 1 / mueff) / ((n + 2) ** 2 + mueff))
        damps = 1 + 2 * max(0, np.sqrt((mueff - 1) / (n + 1)) - 1) + cs
        chi_n = np.sqrt(n) * (1 - 1 / (4 * n) + 1 / (21 * n * n))

        mean = 0.5 * (self.lb + self.ub)
        sigma = self.sigma0 * np.mean(self.span)
        C = np.eye(n)
        B = np.eye(n)
        Dvals = np.ones(n)
        invsqrtC = np.eye(n)
        pc = np.zeros(n)
        ps = np.zeros(n)

        for g in range(self.G):
            A = B @ np.diag(Dvals)
            arz = self.rng.normal(size=(lam, n))
            ary = arz @ A.T
            arx = self.clip(mean + sigma * ary)

            pen, obj, fea, ti = self.evaluate_population(arx)
            self.update_bests(arx, pen, obj, fea, ti)

            idx = np.argsort(pen)
            x_sel = arx[idx[:mu]]
            z_sel = arz[idx[:mu]]
            y_sel = ary[idx[:mu]]

            old_mean = mean.copy()
            mean = np.sum(weights[:, None] * x_sel, axis=0)
            zmean = np.sum(weights[:, None] * z_sel, axis=0)
            ymean = np.sum(weights[:, None] * y_sel, axis=0)

            ps = (1 - cs) * ps + np.sqrt(cs * (2 - cs) * mueff) * (invsqrtC @ ((mean - old_mean) / max(sigma, 1e-16)))
            norm_ps = np.linalg.norm(ps)
            hsig_cond = norm_ps / np.sqrt(1 - (1 - cs) ** (2 * (g + 1))) / chi_n
            hsig = 1.0 if hsig_cond < (1.4 + 2 / (n + 1)) else 0.0
            pc = (1 - cc) * pc + hsig * np.sqrt(cc * (2 - cc) * mueff) * ((mean - old_mean) / max(sigma, 1e-16))

            artmp = (x_sel - old_mean) / max(sigma, 1e-16)
            C = (1 - c1 - cmu) * C + c1 * (np.outer(pc, pc) + (1 - hsig) * cc * (2 - cc) * C)
            for wi, yi in zip(weights, artmp):
                C += cmu * wi * np.outer(yi, yi)
            C = 0.5 * (C + C.T)

            sigma *= np.exp((cs / damps) * (norm_ps / chi_n - 1))
            sigma = max(sigma, 1e-12)

            eigvals, eigvecs = np.linalg.eigh(C)
            eigvals = np.maximum(eigvals, 1e-20)
            Dvals = np.sqrt(eigvals)
            B = eigvecs
            invsqrtC = B @ np.diag(1.0 / Dvals) @ B.T

            self.maybe_store_curve(g)

        return self.finalize_result()


# =========================================================
# L-SHADE (compact self-contained variant)
# =========================================================

class LSHADE(OptimizerBase):
    def __init__(self, problem, P=100, G=1000, seed=123, memory_size=6, pbest_rate=0.11, min_pop_size=4):
        super().__init__(problem, algorithm_name="LSHADE", P=P, G=G, seed=seed)
        self.memory_size = memory_size
        self.pbest_rate = pbest_rate
        self.min_pop_size = min_pop_size

    def _sample_F(self, mu_F):
        while True:
            F = mu_F + 0.1 * np.tan(np.pi * (self.rng.random() - 0.5))
            if F > 0:
                return min(F, 1.0)

    def _sample_CR(self, mu_CR):
        cr = self.rng.normal(mu_CR, 0.1)
        return float(np.clip(cr, 0.0, 1.0))

    def opt(self):
        NP_init = self.P
        NP = self.P
        D = self.D
        H = self.memory_size
        M_F = np.full(H, 0.5, dtype=float)
        M_CR = np.full(H, 0.5, dtype=float)
        memory_idx = 0

        pop = self.rng.uniform(self.lb, self.ub, size=(NP, D))
        pen, obj, fea, ti = self.evaluate_population(pop)
        self.update_bests(pop, pen, obj, fea, ti)
        archive = np.empty((0, D), dtype=float)
        self.maybe_store_curve(0)

        for g in range(1, self.G):
            order = np.argsort(pen)
            pop = pop[order]
            pen = pen[order]
            obj = obj[order]
            fea = fea[order]
            ti = ti[order]

            p_num = max(2, int(np.ceil(self.pbest_rate * NP)))
            p_num = min(p_num, NP)

            F_list = np.zeros(NP, dtype=float)
            CR_list = np.zeros(NP, dtype=float)
            trial = np.zeros_like(pop)

            all_pool = np.vstack([pop, archive]) if len(archive) > 0 else pop.copy()

            for i in range(NP):
                r = self.rng.integers(0, H)
                Fi = self._sample_F(M_F[r])
                CRi = self._sample_CR(M_CR[r])
                F_list[i] = Fi
                CR_list[i] = CRi

                pbest_idx = self.rng.integers(0, p_num)
                xpbest = pop[pbest_idx]

                idx_choices = np.delete(np.arange(NP), i)
                r1 = self.rng.choice(idx_choices)
                xr1 = pop[r1]

                pool_indices = np.arange(all_pool.shape[0])
                invalid = {i, r1}
                valid_pool = [idx for idx in pool_indices if idx not in invalid]
                if len(valid_pool) == 0:
                    xr2 = pop[self.rng.integers(0, NP)]
                else:
                    xr2 = all_pool[self.rng.choice(valid_pool)]

                mutant = pop[i] + Fi * (xpbest - pop[i]) + Fi * (xr1 - xr2)
                mutant = self.clip(mutant)

                jrand = self.rng.integers(0, D)
                cross_mask = self.rng.random(D) < CRi
                cross_mask[jrand] = True
                trial[i] = np.where(cross_mask, mutant, pop[i])

            t_pen, t_obj, t_fea, t_ti = self.evaluate_population(trial)
            self.update_bests(trial, t_pen, t_obj, t_fea, t_ti)

            S_F = []
            S_CR = []
            delta_f = []
            next_pop = []
            next_pen = []
            next_obj = []
            next_fea = []
            next_ti = []

            for i in range(NP):
                if t_pen[i] <= pen[i]:
                    archive = np.vstack([archive, pop[i].copy()])
                    next_pop.append(trial[i].copy())
                    next_pen.append(t_pen[i])
                    next_obj.append(t_obj[i])
                    next_fea.append(t_fea[i])
                    next_ti.append(t_ti[i].copy())
                    improvement = abs(pen[i] - t_pen[i])
                    if improvement > 0:
                        S_F.append(F_list[i])
                        S_CR.append(CR_list[i])
                        delta_f.append(improvement)
                else:
                    next_pop.append(pop[i].copy())
                    next_pen.append(pen[i])
                    next_obj.append(obj[i])
                    next_fea.append(fea[i])
                    next_ti.append(ti[i].copy())

            pop = np.array(next_pop, dtype=float)
            pen = np.array(next_pen, dtype=float)
            obj = np.array(next_obj, dtype=float)
            fea = np.array(next_fea, dtype=bool)
            ti = np.array(next_ti, dtype=float)
            NP = len(pop)

            if archive.shape[0] > NP_init:
                keep = self.rng.choice(np.arange(archive.shape[0]), size=NP_init, replace=False)
                archive = archive[keep]

            if len(S_F) > 0:
                w = np.array(delta_f, dtype=float)
                w = w / np.sum(w)
                S_F = np.array(S_F, dtype=float)
                S_CR = np.array(S_CR, dtype=float)
                new_MF = np.sum(w * (S_F ** 2)) / max(np.sum(w * S_F), 1e-16)
                new_MCR = np.sum(w * S_CR)
                M_F[memory_idx] = np.clip(new_MF, 1e-6, 1.0)
                M_CR[memory_idx] = np.clip(new_MCR, 0.0, 1.0)
                memory_idx = (memory_idx + 1) % H

            NP_target = int(round(self.min_pop_size + (NP_init - self.min_pop_size) * (1 - g / max(self.G - 1, 1))))
            NP_target = max(self.min_pop_size, min(NP_target, len(pop)))
            if len(pop) > NP_target:
                order = np.argsort(pen)
                pop = pop[order[:NP_target]]
                pen = pen[order[:NP_target]]
                obj = obj[order[:NP_target]]
                fea = fea[order[:NP_target]]
                ti = ti[order[:NP_target]]
                NP = NP_target
                if archive.shape[0] > NP_init:
                    archive = archive[:NP_init]

            self.maybe_store_curve(g)

        return self.finalize_result()


# =========================================================
# GENERIC RUNNERS
# =========================================================

def get_scheme_display_name(scheme):
    return SCHEME_DISPLAY_NAMES.get(scheme, scheme)


def get_algorithm_display_name(algorithm_key):
    return ALGORITHM_DISPLAY_NAMES.get(algorithm_key, algorithm_key)


def get_best_feasible_run(results):
    feasible_results = [r for r in results if r["feasible"]]
    if len(feasible_results) == 0:
        return None
    return min(feasible_results, key=lambda x: x["best_fitness"])


def summarize_results(all_results, method_key=None, display_name=None):
    feasible_results = [r for r in all_results if r["feasible"]]
    total_runs = len(all_results)
    best_run = get_best_feasible_run(all_results)
    if len(feasible_results) == 0:
        return {
            "method_key": method_key,
            "display_name": display_name if display_name is not None else method_key,
            "feasible_runs": 0,
            "total_runs": total_runs,
            "mean_best": np.nan,
            "std_best": np.nan,
            "min_best": np.nan,
            "max_best": np.nan,
            "median_best": np.nan,
            "best_run": None,
            "all_feasible_values": np.array([], dtype=float),
        }
    vals = np.array([r["best_fitness"] for r in feasible_results], dtype=float)
    return {
        "method_key": method_key,
        "display_name": display_name if display_name is not None else method_key,
        "feasible_runs": len(feasible_results),
        "total_runs": total_runs,
        "mean_best": np.mean(vals),
        "std_best": np.std(vals),
        "min_best": np.min(vals),
        "max_best": np.max(vals),
        "median_best": np.median(vals),
        "best_run": best_run,
        "all_feasible_values": vals,
    }


def run_multiple_experiments(
    variant="SGuA2",
    init_scheme="rand",
    r_mode="vector",
    n_runs=30,
    P=100,
    G=1000,
    penalty_weight=1e6,
    base_seed=123,
    use_orc_guide_seed=False,
):
    problem = ORCProblem(penalty_weight=penalty_weight)
    all_results = []
    for run_id in range(n_runs):
        seed = base_seed + run_id
        opt = SGuA(
            problem=problem,
            variant=variant,
            P=P,
            G=G,
            branch_mode="linear",
            similarity_tol=1e-3,
            use_orc_guide_seed=use_orc_guide_seed,
            init_scheme=init_scheme,
            r_mode=r_mode,
            seed=seed,
        )
        result = opt.opt()
        result["run_id"] = run_id + 1
        result["seed"] = seed
        all_results.append(result)
        if result["feasible"]:
            print(f"Run {run_id + 1:02d} | {variant} | {init_scheme} | Feasible: True | Best Fitness (sum_ti): {result['best_fitness']:.6f}")
        else:
            print(f"Run {run_id + 1:02d} | {variant} | {init_scheme} | Feasible: False | No feasible solution found")
    return problem, all_results


def run_cmaes_experiments(n_runs=30, P=100, G=1000, penalty_weight=1e6, base_seed=123):
    problem = ORCProblem(penalty_weight=penalty_weight)
    all_results = []
    for run_id in range(n_runs):
        seed = base_seed + run_id
        opt = CMAES(problem=problem, P=P, G=G, seed=seed)
        result = opt.opt()
        result["run_id"] = run_id + 1
        result["seed"] = seed
        all_results.append(result)
        if result["feasible"]:
            print(f"Run {run_id + 1:02d} | CMA-ES | Feasible: True | Best Fitness (sum_ti): {result['best_fitness']:.6f}")
        else:
            print(f"Run {run_id + 1:02d} | CMA-ES | Feasible: False | No feasible solution found")
    return problem, all_results


def run_lshade_experiments(n_runs=30, P=100, G=1000, penalty_weight=1e6, base_seed=123):
    problem = ORCProblem(penalty_weight=penalty_weight)
    all_results = []
    for run_id in range(n_runs):
        seed = base_seed + run_id
        opt = LSHADE(problem=problem, P=P, G=G, seed=seed)
        result = opt.opt()
        result["run_id"] = run_id + 1
        result["seed"] = seed
        all_results.append(result)
        if result["feasible"]:
            print(f"Run {run_id + 1:02d} | L-SHADE | Feasible: True | Best Fitness (sum_ti): {result['best_fitness']:.6f}")
        else:
            print(f"Run {run_id + 1:02d} | L-SHADE | Feasible: False | No feasible solution found")
    return problem, all_results


def run_all_initializations_original_sgua(
    variant="SGuA2",
    r_mode="vector",
    n_runs=30,
    P=100,
    G=1000,
    penalty_weight=1e6,
    base_seed=123,
    use_orc_guide_seed=False,
):
    init_schemes = [
        "rand",
        "uniform",
        "beta_3_2",
        "beta_2_5_2_5",
        "lognorm_0_0_5",
        "lognorm_0_0_67",
        "exprnd_0_5",
        "rayleigh_0_4",
        "weibull_1_1",
        "lhsdesign",
    ]

    all_experiments = {}
    summaries = []
    problem_ref = None

    print("\n" + "=" * 100)
    print(f"ORIGINAL SGuA SOWING + DIFFERENT r SCHEMES ({variant})")
    print("=" * 100)

    for scheme in init_schemes:
        print("\n" + "-" * 100)
        print(f"Init scheme inside ORIGINAL SGuA sowing: {scheme}")
        print("-" * 100)
        problem, results = run_multiple_experiments(
            variant=variant,
            init_scheme=scheme,
            r_mode=r_mode,
            n_runs=n_runs,
            P=P,
            G=G,
            penalty_weight=penalty_weight,
            base_seed=base_seed,
            use_orc_guide_seed=use_orc_guide_seed,
        )
        summary = summarize_results(results, method_key=scheme, display_name=get_scheme_display_name(scheme))
        all_experiments[scheme] = results
        summaries.append(summary)
        problem_ref = problem
        mean_str = f"{summary['mean_best']:.6f}" if np.isfinite(summary["mean_best"]) else "nan"
        std_str = f"{summary['std_best']:.6f}" if np.isfinite(summary["std_best"]) else "nan"
        min_str = f"{summary['min_best']:.6f}" if np.isfinite(summary["min_best"]) else "nan"
        max_str = f"{summary['max_best']:.6f}" if np.isfinite(summary["max_best"]) else "nan"
        print(f"Summary | {scheme} | Feasible: {summary['feasible_runs']}/{summary['total_runs']} | Mean: {mean_str} | Std: {std_str} | Min: {min_str} | Max: {max_str}")
    return problem_ref, all_experiments, summaries


def choose_best_sgua_scheme(summaries):
    valid = [s for s in summaries if np.isfinite(s["mean_best"])]
    if len(valid) == 0:
        return None
    return min(valid, key=lambda s: s["mean_best"])["method_key"]


# =========================================================
# CSV EXPORT HELPERS (INTEGRATED)
# =========================================================

def build_runs_dataframe(results):
    rows = []
    for r in results:
        row = {
            "run_id": r["run_id"],
            "seed": r["seed"],
            "feasible": r["feasible"],
            "best_fitness": r["best_fitness"],
            "penalized_fitness": r["penalized_fitness"],
            "function_evaluations": r["function_evaluations"],
        }
        for i in range(6):
            row[f"td_{i+1}"] = r["td"][i]
            row[f"Ip_{i+1}"] = r["Ip"][i]
            row[f"Ti_{i+1}"] = r["Ti"][i]
        rows.append(row)
    return pd.DataFrame(rows)


def build_convergence_dataframe(results):
    curves = np.array([r["feasible_curve"] for r in results], dtype=float)
    df = pd.DataFrame(curves.T)
    df.index = np.arange(1, curves.shape[1] + 1)
    df.index.name = "iteration"
    df.columns = [f"run_{i+1}" for i in range(curves.shape[0])]
    return df


def save_method_outputs(results, run_csv_name, conv_csv_name):
    runs_df = build_runs_dataframe(results)
    conv_df = build_convergence_dataframe(results)
    runs_df.to_csv(run_csv_name, index=False, encoding="utf-8-sig")
    conv_df.to_csv(conv_csv_name, encoding="utf-8-sig")
    print(f"Saved run results: {run_csv_name}")
    print(f"Saved convergence curves: {conv_csv_name}")


def save_rand_baseline_outputs(all_experiments):
    if "rand" in all_experiments:
        save_method_outputs(
            all_experiments["rand"],
            "sgua_orc_run_results.csv",
            "sgua_orc_feasible_curves.csv",
        )


def run_method_comparison_with_csv(best_sgua_scheme, n_runs=30, P=100, G=1000, penalty_weight=1e6, base_seed=123, r_mode="vector"):
    method_experiments = {}
    method_summaries = []
    problem_ref = None

    print("\n" + "=" * 100)
    print("COMPARATIVE ORC EVALUATION: BEST SGuA VARIANT vs CMA-ES vs L-SHADE")
    print("=" * 100)

    print("\n" + "-" * 100)
    print(f"Selected SGuA baseline for comparison: {best_sgua_scheme}")
    print("-" * 100)
    problem_ref, sgua_results = run_multiple_experiments(
        variant="SGuA2",
        init_scheme=best_sgua_scheme,
        r_mode=r_mode,
        n_runs=n_runs,
        P=P,
        G=G,
        penalty_weight=penalty_weight,
        base_seed=base_seed,
        use_orc_guide_seed=False,
    )
    method_experiments[f"SGuA2-{best_sgua_scheme}"] = sgua_results
    method_summaries.append(summarize_results(sgua_results, method_key=f"SGuA2-{best_sgua_scheme}", display_name=f"SGuA2 ({get_scheme_display_name(best_sgua_scheme)})"))
    save_method_outputs(sgua_results, "lhsdesign_orc_run_results.csv", "lhsdesign_orc_feasible_curves.csv")

    print("\n" + "-" * 100)
    print("CMA-ES runs")
    print("-" * 100)
    _, cma_results = run_cmaes_experiments(n_runs=n_runs, P=P, G=G, penalty_weight=penalty_weight, base_seed=base_seed)
    method_experiments["CMAES"] = cma_results
    method_summaries.append(summarize_results(cma_results, method_key="CMAES", display_name=get_algorithm_display_name("CMAES")))
    save_method_outputs(cma_results, "cmaes_orc_run_results.csv", "cmaes_orc_feasible_curves.csv")

    print("\n" + "-" * 100)
    print("L-SHADE runs")
    print("-" * 100)
    _, lshade_results = run_lshade_experiments(n_runs=n_runs, P=P, G=G, penalty_weight=penalty_weight, base_seed=base_seed)
    method_experiments["LSHADE"] = lshade_results
    method_summaries.append(summarize_results(lshade_results, method_key="LSHADE", display_name=get_algorithm_display_name("LSHADE")))
    save_method_outputs(lshade_results, "lshade_orc_run_results.csv", "lshade_orc_feasible_curves.csv")

    return problem_ref, method_experiments, method_summaries


# =========================================================
# TABLES AND STATS
# =========================================================

def format_relay_triplet(td, ip, ti):
    return f"{td:.4f} / {ip:.2f} / {ti:.4f}"


def build_publication_table(summaries, first_col_name="Distribution Method"):
    rows = []
    for summary in summaries:
        best_run = summary.get("best_run", None)
        row = {first_col_name: summary["display_name"], "Total ti": np.nan if best_run is None else float(best_run["best_fitness"])}
        if best_run is None:
            for i in range(6):
                row[f"Relay-{i+1} (td, Ip, ti)"] = "NO FEASIBLE SOLUTION"
        else:
            for i in range(6):
                row[f"Relay-{i+1} (td, Ip, ti)"] = format_relay_triplet(best_run["td"][i], best_run["Ip"][i], best_run["Ti"][i])
        rows.append(row)
    cols = [first_col_name] + [f"Relay-{i+1} (td, Ip, ti)" for i in range(6)] + ["Total ti"]
    return pd.DataFrame(rows, columns=cols)


def print_publication_table(publication_df, title_line, first_col_name="Distribution Method"):
    print("\n" + "=" * 180)
    print(title_line)
    print("=" * 180)
    header = (
        f"{first_col_name:<24}"
        f"{'Relay-1 (td, Ip, ti)':<28}"
        f"{'Relay-2 (td, Ip, ti)':<28}"
        f"{'Relay-3 (td, Ip, ti)':<28}"
        f"{'Relay-4 (td, Ip, ti)':<28}"
        f"{'Relay-5 (td, Ip, ti)':<28}"
        f"{'Relay-6 (td, Ip, ti)':<28}"
        f"{'Total ti':<12}"
    )
    print(header)
    print("-" * 180)
    for _, row in publication_df.iterrows():
        total_ti = row["Total ti"]
        total_str = f"{total_ti:.4f}" if np.isfinite(total_ti) else "nan"
        line = f"{row[first_col_name]:<24}"
        for i in range(6):
            line += f"{row[f'Relay-{i+1} (td, Ip, ti)']:<28}"
        line += total_str
        print(line)


def get_paired_feasible_values(results_ref, results_cmp):
    ref_map = {r["run_id"]: r["best_fitness"] for r in results_ref if r["feasible"]}
    cmp_map = {r["run_id"]: r["best_fitness"] for r in results_cmp if r["feasible"]}
    common_run_ids = sorted(set(ref_map.keys()).intersection(cmp_map.keys()))
    if len(common_run_ids) == 0:
        return common_run_ids, np.array([], dtype=float), np.array([], dtype=float)
    ref_vals = np.array([ref_map[rid] for rid in common_run_ids], dtype=float)
    cmp_vals = np.array([cmp_map[rid] for rid in common_run_ids], dtype=float)
    return common_run_ids, ref_vals, cmp_vals


def bonferroni_correction(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    if m == 0:
        return p_values
    return np.minimum(1.0, p_values * m)


def compute_rank_biserial_from_differences(differences):
    differences = np.asarray(differences, dtype=float)
    non_zero = differences[np.abs(differences) > 1e-15]
    if non_zero.size == 0:
        return 0.0, 0.0, 0.0, 0
    abs_diff = np.abs(non_zero)
    ranks = rankdata(abs_diff, method="average")
    w_pos = np.sum(ranks[non_zero > 0])
    w_neg = np.sum(ranks[non_zero < 0])
    rbc = (w_pos - w_neg) / (w_pos + w_neg)
    return float(rbc), float(w_pos), float(w_neg), int(non_zero.size)


def effect_size_label(abs_effect):
    if abs_effect < 0.10:
        return "negligible"
    if abs_effect < 0.30:
        return "small"
    if abs_effect < 0.50:
        return "medium"
    return "large"


def build_wilcoxon_statistics_table(all_experiments, summaries, reference_method=None, alpha=0.05):
    valid_summaries = [s for s in summaries if np.isfinite(s["mean_best"])]
    if len(valid_summaries) < 2:
        return pd.DataFrame()
    if reference_method is None:
        reference_method = min(valid_summaries, key=lambda x: x["mean_best"])["method_key"]

    comparisons = []
    raw_p_values = []
    for summary in valid_summaries:
        comparator = summary["method_key"]
        if comparator == reference_method:
            continue
        _, ref_vals, cmp_vals = get_paired_feasible_values(all_experiments[reference_method], all_experiments[comparator])
        if len(ref_vals) < 3:
            comparisons.append({
                "reference_method": reference_method,
                "comparator_method": comparator,
                "reference_display": next(s["display_name"] for s in valid_summaries if s["method_key"] == reference_method),
                "comparator_display": summary["display_name"],
                "n_pairs": len(ref_vals),
                "ref_mean": np.nan if len(ref_vals) == 0 else float(np.mean(ref_vals)),
                "cmp_mean": np.nan if len(cmp_vals) == 0 else float(np.mean(cmp_vals)),
                "median_cmp_minus_ref": np.nan,
                "wilcoxon_stat": np.nan,
                "p_raw": np.nan,
                "rank_biserial": np.nan,
                "effect_magnitude": "insufficient_data",
                "decision": "Insufficient paired feasible runs for Wilcoxon test",
            })
            raw_p_values.append(np.nan)
            continue
        differences = cmp_vals - ref_vals
        test = wilcoxon(ref_vals, cmp_vals, zero_method="wilcox", alternative="two-sided", method="auto")
        rbc, w_pos, w_neg, effective_n = compute_rank_biserial_from_differences(differences)
        comparisons.append({
            "reference_method": reference_method,
            "comparator_method": comparator,
            "reference_display": next(s["display_name"] for s in valid_summaries if s["method_key"] == reference_method),
            "comparator_display": summary["display_name"],
            "n_pairs": int(len(ref_vals)),
            "ref_mean": float(np.mean(ref_vals)),
            "cmp_mean": float(np.mean(cmp_vals)),
            "median_cmp_minus_ref": float(np.median(differences)),
            "wilcoxon_stat": float(test.statistic),
            "p_raw": float(test.pvalue),
            "rank_biserial": rbc,
            "w_positive": w_pos,
            "w_negative": w_neg,
            "effective_n": effective_n,
            "effect_magnitude": effect_size_label(abs(rbc)),
            "decision": "",
        })
        raw_p_values.append(float(test.pvalue))

    raw_array = np.array(raw_p_values, dtype=float)
    adjusted = np.full_like(raw_array, np.nan, dtype=float)
    valid_mask = np.isfinite(raw_array)
    adjusted[valid_mask] = bonferroni_correction(raw_array[valid_mask])

    for row, p_adj in zip(comparisons, adjusted):
        row["p_bonferroni"] = np.nan if not np.isfinite(p_adj) else float(p_adj)
        row["significant_after_bonferroni"] = bool(np.isfinite(p_adj) and (p_adj < alpha))
        if not np.isfinite(row["p_raw"]):
            row["decision"] = "Insufficient paired feasible runs for Wilcoxon test"
        elif row["significant_after_bonferroni"]:
            if row["median_cmp_minus_ref"] > 0:
                row["decision"] = f"{row['reference_display']} is significantly better than {row['comparator_display']} after Bonferroni correction"
            elif row["median_cmp_minus_ref"] < 0:
                row["decision"] = f"{row['comparator_display']} is significantly better than {row['reference_display']} after Bonferroni correction"
            else:
                row["decision"] = "Statistically significant after Bonferroni correction, but median difference is zero"
        else:
            row["decision"] = "No statistically significant difference after Bonferroni correction"

    return pd.DataFrame(comparisons)


def print_wilcoxon_statistics_table(stats_df):
    if stats_df.empty:
        print("\nWilcoxon analysis could not be produced.")
        return
    print("\n" + "=" * 170)
    print("WILCOXON SIGNED-RANK TEST RESULTS (BONFERRONI-CORRECTED)")
    print("=" * 170)
    print(
        f"{'Reference':<22}"
        f"{'Comparator':<22}"
        f"{'Pairs':<8}"
        f"{'Ref Mean':<12}"
        f"{'Cmp Mean':<12}"
        f"{'Median(C-R)':<14}"
        f"{'p raw':<12}"
        f"{'p Bonf.':<12}"
        f"{'Sig.':<8}"
        f"{'RBC':<10}"
        f"{'Effect':<12}"
        f"Conclusion"
    )
    for _, row in stats_df.iterrows():
        ref_mean = f"{row['ref_mean']:.6f}" if np.isfinite(row["ref_mean"]) else "nan"
        cmp_mean = f"{row['cmp_mean']:.6f}" if np.isfinite(row["cmp_mean"]) else "nan"
        median_diff = f"{row['median_cmp_minus_ref']:.6f}" if np.isfinite(row["median_cmp_minus_ref"]) else "nan"
        p_raw = f"{row['p_raw']:.6g}" if np.isfinite(row["p_raw"]) else "nan"
        p_b = f"{row['p_bonferroni']:.6g}" if np.isfinite(row["p_bonferroni"]) else "nan"
        sig = "Yes" if bool(row["significant_after_bonferroni"]) else "No"
        rbc = f"{row['rank_biserial']:.4f}" if np.isfinite(row["rank_biserial"]) else "nan"
        print(
            f"{row['reference_display']:<22}"
            f"{row['comparator_display']:<22}"
            f"{int(row['n_pairs']):<8}"
            f"{ref_mean:<12}"
            f"{cmp_mean:<12}"
            f"{median_diff:<14}"
            f"{p_raw:<12}"
            f"{p_b:<12}"
            f"{sig:<8}"
            f"{rbc:<10}"
            f"{row['effect_magnitude']:<12}"
            f"{row['decision']}"
        )


# =========================================================
# PRINT HELPERS
# =========================================================

def print_result_table(title, result):
    print("\n" + title)
    print(f"{'Relay':<10}{'td':<12}{'Ip(A)':<12}{'Ti(s)':<12}")
    for i in range(6):
        print(f"{'Relay-' + str(i+1):<10}{result['td'][i]:<12.6f}{result['Ip'][i]:<12.3f}{result['Ti'][i]:<12.6f}")


def print_constraint_check(problem, result):
    cc = problem.check_constraints_from_ti(result["Ti"])
    print("\nConstraint Check:")
    print(f"1 <= t1 <= 2.2 : {cc['t1_range']}")
    print(f"1 <= t2 <= 2.2 : {cc['t2_range']}")
    print(f"1 <= t3 <= 2.2 : {cc['t3_range']}")
    print(f"1 <= t4 <= 2.2 : {cc['t4_range']}")
    print(f"1 <= t5 <= 2.2 : {cc['t5_range']}")
    print(f"1 <= t6 <= 2.2 : {cc['t6_range']}")
    print(f"t1 - t2 >= 0.3 : {cc['c12_ok']}   value = {cc['t1_minus_t2']:.6f}")
    print(f"t2 - t3 >= 0.3 : {cc['c23_ok']}   value = {cc['t2_minus_t3']:.6f}")
    print(f"t2 - t4 >= 0.3 : {cc['c24_ok']}   value = {cc['t2_minus_t4']:.6f}")
    print(f"t2 - t5 >= 0.3 : {cc['c25_ok']}   value = {cc['t2_minus_t5']:.6f}")
    print(f"t2 - t6 >= 0.3 : {cc['c26_ok']}   value = {cc['t2_minus_t6']:.6f}")


def print_summary_table(summaries, label_col="Method"):
    valid = [s for s in summaries if np.isfinite(s["mean_best"])]
    invalid = [s for s in summaries if not np.isfinite(s["mean_best"])]
    rows = sorted(valid, key=lambda x: x["mean_best"]) + invalid
    print("\n" + "=" * 120)
    print("SUMMARY TABLE")
    print("=" * 120)
    print(f"{'Rank':<6}{label_col:<26}{'Feasible':<12}{'Mean':<14}{'Std':<14}{'Min':<14}{'Max':<14}")
    for i, s in enumerate(rows, start=1):
        mean_str = f"{s['mean_best']:.6f}" if np.isfinite(s["mean_best"]) else "nan"
        std_str = f"{s['std_best']:.6f}" if np.isfinite(s["std_best"]) else "nan"
        min_str = f"{s['min_best']:.6f}" if np.isfinite(s["min_best"]) else "nan"
        max_str = f"{s['max_best']:.6f}" if np.isfinite(s["max_best"]) else "nan"
        feasible_str = f"{s['feasible_runs']}/{s['total_runs']}"
        print(f"{i:<6}{s['display_name']:<26}{feasible_str:<12}{mean_str:<14}{std_str:<14}{min_str:<14}{max_str:<14}")


def print_best_scheme_details(problem, experiments, summaries):
    valid = [s for s in summaries if np.isfinite(s["mean_best"])]
    if len(valid) == 0:
        print("\nNo feasible method found.")
        return
    best_summary = min(valid, key=lambda x: x["mean_best"])
    best_key = best_summary["method_key"]
    best_run = min([r for r in experiments[best_key] if r["feasible"]], key=lambda x: x["best_fitness"])
    print("\n" + "=" * 110)
    print(f"BEST METHOD BY MEAN PERFORMANCE: {best_summary['display_name']}")
    print("=" * 110)
    print_result_table(f"Best feasible run for {best_summary['display_name']}", best_run)
    print_constraint_check(problem, best_run)
    print(f"\nBest fitness (sum_ti): {best_run['best_fitness']:.6f}")
    print(f"Function evaluations: {best_run['function_evaluations']}")


# =========================================================
# PLOTS
# =========================================================

def plot_experiments_boxplot(experiments, label_lookup, title, figsize=(12, 6)):
    labels = []
    data = []

    for key, results in experiments.items():
        vals = [r["best_fitness"] for r in results if r["feasible"]]
        if len(vals) > 0:
            labels.append(f"{label_lookup.get(key, key)}\n({len(vals)}/{len(results)})")
            data.append(vals)

    if len(data) == 0:
        print("No feasible results found for plotting.")
        return

    plt.figure(figsize=figsize)
    plt.boxplot(data, tick_labels=labels, showmeans=True)
    plt.xticks(rotation=20, ha="right")
    plt.ylabel("Best feasible objective (sum_ti)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_experiments_convergence(experiments, label_lookup, title, show_std=True, figsize=(12, 6)):
    plt.figure(figsize=figsize)
    plotted = False

    for key, results in experiments.items():
        curves = np.array([r["feasible_curve"] for r in results], dtype=float)

        if curves.size == 0 or not np.any(np.isfinite(curves)):
            continue

        mean_curve = np.nanmean(curves, axis=0)
        std_curve = np.nanstd(curves, axis=0)
        iterations = np.arange(1, curves.shape[1] + 1)

        plt.plot(iterations, mean_curve, linewidth=2.5, label=label_lookup.get(key, key))

        if show_std:
            lower = mean_curve - std_curve
            upper = mean_curve + std_curve
            plt.fill_between(iterations, lower, upper, alpha=0.15)

        plotted = True

    if not plotted:
        print("No feasible convergence curves found.")
        return

    plt.xlabel("Iteration")
    plt.ylabel("Mean best feasible objective (sum_ti)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_single_method_convergence(results, title, figsize=(12, 6)):
    curves = np.array([r["feasible_curve"] for r in results], dtype=float)

    if curves.size == 0 or not np.any(np.isfinite(curves)):
        print("No feasible convergence curves found.")
        return

    iterations = np.arange(1, curves.shape[1] + 1)
    mean_curve = np.nanmean(curves, axis=0)
    std_curve = np.nanstd(curves, axis=0)

    plt.figure(figsize=figsize)

    for c in curves:
        plt.plot(iterations, c, alpha=0.12, linewidth=1)

    plt.plot(iterations, mean_curve, linewidth=3, label="Mean feasible curve")
    plt.fill_between(iterations, mean_curve - std_curve, mean_curve + std_curve, alpha=0.18)

    plt.xlabel("Iteration")
    plt.ylabel("Best feasible objective (sum_ti)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":
    SGUA_VARIANT = "SGuA2"
    N_RUNS = 30
    POP_SIZE = 100
    ITERATIONS = 1000
    PENALTY_WEIGHT = 1e6
    BASE_SEED = 123
    USE_ORC_GUIDE_SEED = False
    R_MODE = "vector"

    # 1) SGuA initialization study
    problem, all_experiments, summaries = run_all_initializations_original_sgua(
        variant=SGUA_VARIANT,
        r_mode=R_MODE,
        n_runs=N_RUNS,
        P=POP_SIZE,
        G=ITERATIONS,
        penalty_weight=PENALTY_WEIGHT,
        base_seed=BASE_SEED,
        use_orc_guide_seed=USE_ORC_GUIDE_SEED,
    )

    print_summary_table(summaries, label_col="Init Scheme")
    print_best_scheme_details(problem, all_experiments, summaries)

    publication_df = build_publication_table(summaries, first_col_name="Distribution Method")
    print_publication_table(
        publication_df,
        title_line="OPTIMAL RELAY COORDINATION PARAMETERS (TD, IP, TI) AND TOTAL RUN TIMES\nOBTAINED FROM THE ISGUA / SGUA ALGORITHM USING DIFFERENT INITIAL POPULATION METHODS",
        first_col_name="Distribution Method",
    )
    publication_df.to_csv("sgua_orc_publication_table.csv", index=False, encoding="utf-8-sig")

    stats_df = build_wilcoxon_statistics_table(all_experiments=all_experiments, summaries=summaries, reference_method=None, alpha=0.05)
    print_wilcoxon_statistics_table(stats_df)
    stats_df.to_csv("sgua_orc_wilcoxon_bonferroni.csv", index=False, encoding="utf-8-sig")

    # CSV outputs for original random baseline
    save_rand_baseline_outputs(all_experiments)

    init_label_lookup = {k: get_scheme_display_name(k) for k in all_experiments.keys()}
    plot_experiments_boxplot(all_experiments, init_label_lookup, title=f"Original SGuA sowing + different r schemes - {SGUA_VARIANT}")
    plot_experiments_convergence(all_experiments, init_label_lookup, title=f"Mean convergence comparison - Original SGuA sowing ({SGUA_VARIANT})")

    # 2) Best SGuA variant vs CMA-ES vs L-SHADE
    best_sgua_scheme = choose_best_sgua_scheme(summaries)
    comp_problem, comp_experiments, comp_summaries = run_method_comparison_with_csv(
        best_sgua_scheme=best_sgua_scheme,
        n_runs=N_RUNS,
        P=POP_SIZE,
        G=ITERATIONS,
        penalty_weight=PENALTY_WEIGHT,
        base_seed=BASE_SEED,
        r_mode=R_MODE,
    )

    print_summary_table(comp_summaries, label_col="Algorithm")
    print_best_scheme_details(comp_problem, comp_experiments, comp_summaries)

    comp_publication_df = build_publication_table(comp_summaries, first_col_name="Algorithm")
    print_publication_table(
        comp_publication_df,
        title_line="COMPARATIVE ORC PARAMETERS AND TOTAL RUN TIMES\nBEST SGuA VARIANT vs CMA-ES vs L-SHADE",
        first_col_name="Algorithm",
    )
    comp_publication_df.to_csv("orc_comparative_publication_table_sgua_cmaes_lshade.csv", index=False, encoding="utf-8-sig")

    comp_stats_df = build_wilcoxon_statistics_table(all_experiments=comp_experiments, summaries=comp_summaries, reference_method=None, alpha=0.05)
    print_wilcoxon_statistics_table(comp_stats_df)
    comp_stats_df.to_csv("orc_comparative_wilcoxon_bonferroni_sgua_cmaes_lshade.csv", index=False, encoding="utf-8-sig")

    comp_label_lookup = {s["method_key"]: s["display_name"] for s in comp_summaries}
    plot_experiments_boxplot(comp_experiments, comp_label_lookup, title="ORC comparison: Best SGuA variant vs CMA-ES vs L-SHADE")
    plot_experiments_convergence(comp_experiments, comp_label_lookup, title="Mean convergence comparison: Best SGuA variant vs CMA-ES vs L-SHADE")

    plot_single_method_convergence(
        comp_experiments[f"SGuA2-{best_sgua_scheme}"],
        title=f"Mean convergence behavior over 30 runs - SGuA2 ({get_scheme_display_name(best_sgua_scheme)})"
    )
    plot_single_method_convergence(comp_experiments["CMAES"], title="Mean convergence behavior over 30 runs - CMA-ES")
    plot_single_method_convergence(comp_experiments["LSHADE"], title="Mean convergence behavior over 30 runs - L-SHADE")



ORIGINAL SGuA SOWING + DIFFERENT r SCHEMES (SGuA2)

----------------------------------------------------------------------------------------------------
Init scheme inside ORIGINAL SGuA sowing: rand
----------------------------------------------------------------------------------------------------
Run 01 | SGuA2 | rand | Feasible: True | Best Fitness (sum_ti): 7.678073


KeyboardInterrupt: 